# 石器时代 NLP · 核心架构 (arch/)


本 Notebook 是 `stone-age-NLP` 的核心架构（V1 → V4）。
每个章节对应 arch 包下的一个独立模块文件，可用 `extract_to_classic.py` 提取为纯 .py。

SPDX-License-Identifier: CC-BY-4.0


## config —— 配置


In [ ]:
vocab_headefine = ['<pad>', '<start>', '<end>', '<unk>']
repl = '_'


## test —— 基础词统计


In [ ]:
import jieba
import re

all_in_lst=[vocab_headefine,{},{},set(),dict()]
all_tokens = all_in_lst[3]
word2idx = lambda: all_in_lst[2]
idx2word = lambda: all_in_lst[1]
word_dic=all_in_lst[4]


def fenci(words_str):
    # import jieba
    no_sign_WS = re.sub(r'\W', repl, words_str)
    no_sign_WS2 = re.sub(f'{repl}{repl}+', repl, no_sign_WS)
    cut_res = jieba.lcut(no_sign_WS2)
    # print(cut_res)
    return cut_res

def tongji(cut_res):
    for i in range(1,len(cut_res)):
        dic=word_dic.get(cut_res[i-1],{})
        dic[cut_res[i]]=(dic.get(cut_res[i],0)+1)
        word_dic[cut_res[i-1]]=dic
    # return word_dic

def update_vocab(new_words):
    # global word2idx,idx2word
    all_tokens.update(new_words)
    vocab = vocab_headefine + sorted(all_tokens)
    # print('ok1')
    # print(*enumerate(vocab))
    all_in_lst[1] = dict(enumerate(vocab))
    all_in_lst[2] = {w: i for i, w in idx2word().items()}


def load_from_file(filename):
    words=[]
    with open(filename,mode='r',encoding='gbk',errors='ignore') as file:
        context=file.read()
        words = fenci(context)
        tongji(words)
        update_vocab(words)
    # return words


# jieba.lcut(re.sub(r'\W','\r',context))



## test_func —— 预测工具箱


In [ ]:
import random

sort_WDlst=lambda word_dic,word: sorted(word_dic[word].items(), key=lambda d: d[1], reverse=True)

def predict_next(word_dic,word):
    while not word == repl:
        print(word,end='')
        word = sort_WDlst(word_dic,word)[0][0]
    print(word)

def predict_next_rand(word_dic,word):
    while not word == repl:
        print(word,end='')
        word=random.choices(list(word_dic[word].keys()),weights=list(word_dic[word].values()),k=1)[0]
    print(repl)

def predict_next_rand_r(word_dic,word,l=None):
    import time
    print(word, end='')
    while True:
        lst = sort_WDlst(word_dic,word)
        if type(l) is type(int()):
            lst=lst[:l]
        if type(l) is type(tuple()):
            lst=lst[:random.randint(*l)]
        if type(l) is type(list()):
            lst=lst[l[0]:l[1]]
        dic=dict(lst)
        new_word=random.choices(list(dic.keys()),weights=list(dic.values()),k=1)[0]
        if not new_word == repl:
            word = new_word
            print(word,end='')
        else:
            if len(word_dic[word])==1:
                print(word_dic[word])
                break
            else:
                print('-',end='')

def predict_by_recursion(word_dic,start,end,l):
    if start == end:
        return end
    if start == repl:
        return ''
    lst = sort_WDlst(word_dic,start)
    for i in lst[:l]:
        result=predict_by_recursion(word_dic,i[0],end,l)
        if result and result != repl:
            return start+result

def predict_by_loop(word_dic,start,end,l,BFS:bool=False):
    class W_node_lite():
        def __init__(self,data):
            self.data=data
            self.came_from=[]
            self.next=[]
    lst = [W_node_lite(start)]
    # visited = set()
    all_node=dict()
    while lst:
        cur_node = lst.pop(BFS-1)
        if cur_node not in all_node.keys():
            all_node[cur_node.data]=cur_node
        for wd,t in [*sort_WDlst(word_dic,cur_node.data)[:l]]:
            if wd not in all_node.keys():
                node = W_node_lite(wd)
                node.came_from.append(cur_node)
                # all_node[wd] = node
                if not (cur_node.data == end or wd == repl):
                    lst.append(node)
                    cur_node.next.append(node)
            else:
                node = all_node[wd]
                node.came_from.append(cur_node)
    return all_node

def get_path_from_PBL(all_node,start,end,i):
    res_lst = []
    if i > 0:
        cur_node = all_node[start]
        while True:
            # print(cur_node.data,end='')
            res_lst.append(cur_node.data)
            if cur_node == all_node[end]: return res_lst
            else: cur_node=cur_node.next[i-1]
    if i < 0:
        cur_node = all_node[end]
        while True:
            # print(cur_node.data, end='')
            res_lst.append(cur_node.data)
            if cur_node == all_node[start]:
                res_lst.reverse()
                return res_lst
            else:
                cur_node = cur_node.came_from[-i-1]

    pass




## test2 —— 共现 + 加权


In [ ]:
import jieba
import re


word_dic_2 = dict()

dic_fast_sorted=lambda dic: sorted(dic.items(), key=lambda d: d[1], reverse=True)

def tongji2(cut_res:list): # 统计词和词之间在同一个句子出现的次数，将来可以算概率
    t_lst = []
    for word in cut_res:
        if word != repl:
            t_lst.append(word)
        else:
            # print(t_lst)
            for cur_wd in t_lst:
                dic = word_dic_2.get(cur_wd, {})
                for wd in t_lst:
                    if wd != cur_wd and wd != repl:
                        dic[wd]=dic.get(wd,0)+1
                word_dic_2[cur_wd]=dic
            t_lst=[]

def jiaquan1(input_words:list, word_dic1:dict, word_dic2:dict):
    points=[0,0.5,0.25]
    dic1_plus_res,dic2_plus_res={},{}
    res_dic={}
    for iwd in input_words:
        for k,v in word_dic2[iwd].items():
            dic2_plus_res[k]=dic2_plus_res.get(k,0)+v
            res_dic[k]=0
        for k,v in word_dic1[iwd].items():
            dic1_plus_res[k]=dic1_plus_res.get(k,0)+v
            res_dic[k]=0
    for k in res_dic.keys():
        res_dic[k] = (dic1_plus_res.get(k,0) * points[1]) + (dic2_plus_res.get(k,0) * points[2])
    res_lst = dic_fast_sorted(res_dic)
    return res_lst[:len(res_lst)//2]

def jiaquan_output_test1(words,ignor_words=''):
    lst=[]
    lst.extend(words)
    print(*words,sep='',end='')
    while lst[-1] != repl:
        jiaquan_res=jiaquan1(lst,word_dic,word_dic_2)
        while jiaquan_res[0][0] in ignor_words:
            jiaquan_res.pop(0)
        new_word=jiaquan_res[0][0]
        lst.append(new_word)
        print(new_word,end='')
    print()
    return lst




## test3 —— 词距离权重


In [ ]:


word_dic_3 = dict()

def tongji3(cut_res:list): # 统计词和词之间的距离
    t_lst = []
    for word in cut_res:
        if word != repl:
            t_lst.append(word)
        else:
            # print(t_lst)
            for cur_wd in t_lst:
                dic = word_dic_3.get(cur_wd, {})
                for wd in t_lst:
                    if wd != cur_wd and wd != repl:
                        dic[wd]=dic.get(wd,[float('inf'),0])
                        d=abs(t_lst.index(wd)-t_lst.index(cur_wd))
                        if d < dic[wd][0]: dic[wd][0] = d
                        if d > dic[wd][1]: dic[wd][1] = d
                word_dic_3[cur_wd]=dic
            t_lst=[]

def minmax_norm(t_dic, a):   #归一化：将数值从其它区间比如[1,5878]变成[0,1]，保证权值按权重计算
    for i in range(1,a):
        vals=[v[i] for v in t_dic.values()]
        lo,hi=min(vals),max(vals)
        if lo==hi: continue
        for v in t_dic.values():
            v[i]=(v[i]-lo)/(hi-lo)

def jiaquan2(input_words:list, word_dic1:dict, word_dic2:dict, word_dic3:dict):
    points=[0,0.5,0.25,0.25]
    t_dic={}
    for iwd in input_words:
        for k,v in word_dic1.get(iwd,{}).items():
            t_dic.setdefault(k,[0,0,0,0])[1]+=v
        for k,v in word_dic2.get(iwd,{}).items():
            t_dic.setdefault(k,[0,0,0,0])[2]+=v
        for k,v in word_dic3.get(iwd,{}).items():
            t_dic.setdefault(k,[0,0,0,0])[3]+=1/v[0]
    minmax_norm(t_dic,3)
    for v in t_dic.values():
        v[0]=sum(v[i]*points[i] for i in range(1,4))
    res_lst=sorted(t_dic.items(),key=lambda d:d[1][0],reverse=True)
    return res_lst[:len(res_lst)//2]

def jiaquan_output_test2(words,ignor_words=''):
    lst=[]
    lst.extend(words)
    print(*words,sep='',end='')
    while lst[-1] != repl:
        jiaquan_res=jiaquan2(lst,word_dic,word_dic_2,word_dic_3)
        while jiaquan_res[0][0] in ignor_words or jiaquan_res[0][0] in lst:
            jiaquan_res.pop(0)
        new_word=jiaquan_res[0][0]
        lst.append(new_word)
        print(new_word,end='')
    print()
    return lst




## test4 —— 图结构字典树


In [ ]:
class WdConfig:
    def __init__(self,
                 fast_mode=1,                 # 缓存方式 0, 1, 2
                 mode='X',                    # 模式，B or X (mB(memBer,neighBors) or mX(miX,eXtend))，即无向平等或有向混合
                 builder=None,                # 函数 dic -> any，dic为_links格式
                 weight_aggregator=None):     # 函数 weights_lst -> number
        """
        :param mode: 图模式，'B' 无向（双向），'X' 有向（默认）
        """
        self.fast_mode = fast_mode
        self.mode = mode
        self.builder = builder or self.origin_dic_build
        self.weight_aggregator = weight_aggregator or (lambda lst: sum(lst))

    word_s_build = lambda self,dic: {k.word for k in dic}                     # {词} 集合
    origin_dic_build = lambda self,dic: dic                                   # 原始格式：{node: 边信息}，不转换
    list_weight_builder = lambda self,dic: [(k, self.weight_aggregator(v['weights_lst'])) for k, v in dic.items()]  # [(节点, 聚合权重)]
    dict_weight_builder = lambda self,dic: {k.word: self.weight_aggregator(v['weights_lst']) for k, v in dic.items()}  # {词: 聚合权重}
    list_full_builder = lambda self,dic: [(k.word, v['weights_lst'], v.get('other', {}), v.get('direct', '')) for k, v in dic.items()]  # [(词, 权重列表, 其他, 方向)]
    dict_full_builder = lambda self,dic: {k.word: {
        'weights': v['weights_lst'],
        'other': v.get('other', {}),
        'direct': v.get('direct', '')
    } for k, v in dic.items()}                        # {词: {权重, 其他, 方向}}

    # ---------- 预设配置模板（供用户选用） ----------
    @classmethod
    def words(cls, fast_mode=2, mode='X'):
        """返回 {词} 集合"""
        cfg = cls(fast_mode=fast_mode, mode=mode)
        cfg.builder = cfg.word_s_build
        return cfg

    @classmethod
    def default(cls, fast_mode=1, mode='X', agg=None):
        """返回 [(节点, 聚合权重), ...] 列表"""
        cfg = cls(fast_mode=fast_mode, mode=mode, weight_aggregator=agg)
        cfg.builder = cfg.list_weight_builder
        return cfg

    @classmethod
    def dict_full(cls, fast_mode=0, mode='X'):
        """返回 {词: {"weights": [...], "other": {...}, "direct": "..."}}"""
        cfg = cls(fast_mode=fast_mode, mode=mode)
        cfg.builder = cfg.dict_full_builder
        return cfg

DEFAULT_CONFIG = WdConfig.default()

class Wd_Node:
    def __init__(self, word, config=None):
        self.word = word
        self.config = config or DEFAULT_CONFIG

        # 唯一真相源：邻居节点 -> 边信息
        self._links = {}  # {node_obj: {"weights_lst": [], "other": {}, "direct": ""}}

        if int(self.config.fast_mode)>0:
            self.parents_lst = None
            self.children_lst = None

    def _get_links(self,direct=''):
        if direct=='': direct='pc'
        t_dic={}
        for k,v in self._links.items():
            d=v.get('direct','')
            if self.config.mode=='B' or d in direct:
                t_dic[k]=v
        return self.config.builder(t_dic)

    def get_links(self,direct=''):
        if direct=='': direct='pc'
        fm=int(self.config.fast_mode)
        if fm == 0: return self._get_links(direct)
        if fm > 0:
            if (direct in ('p','pc','') and self.parents_lst is None) or (self.config.mode=='B' or direct in ('c','pc','') and self.children_lst is None):
                self.update_cache()
            if self.config.mode=='B' or direct=='c':
                return self.children_lst
            elif direct=='p':
                return self.parents_lst
            elif direct=='pc':
                return self._merge(self.parents_lst,self.children_lst)

    def add_links(self, node, weights_lst, direct='c'):
        edge = self._links.setdefault(node, {'weights_lst': [], 'other': {}, 'direct': ''})
        edge['weights_lst']=weights_lst
        if self.config.mode!='B':
            edge['direct'] = self._merge_direct(edge['direct'], direct)
        if self.config.fast_mode==2 or self.parents_lst is not None or self.children_lst is not None:
            self.update_cache()
        return edge

    def update_links(self, node, weights_lst=None, direct=None, other=None):
        edge = self._links.get(node)
        if edge is None:
            return None
        if weights_lst:
            edge['weights_lst']=weights_lst
        if direct:
            edge['direct'] = self._merge_direct(edge['direct'], direct)
        if other:
            edge['other'].update(other)
        if self.parents_lst is not None or self.children_lst is not None:
            self.update_cache()
        return edge

    def update_cache(self):
        if int(self.config.fast_mode)==0: return
        if self.config.mode=='B':
            self.children_lst = self._get_links('c')
        else:
            self.parents_lst = self._get_links('p')
            self.children_lst = self._get_links('c')

    def _merge(self,a,b):
        if isinstance(a,set): return a|b
        if isinstance(a,dict): return {**a,**b}
        if isinstance(a,list): return a+b
        return (a,b)

    @staticmethod
    def _merge_direct(d0,d1):
        if not d0: return d1
        if not d1: return d0
        return d0 if d0==d1 else 'pc'



word_index_dic = dict()

from itertools import pairwise

def build_dic_tree_sample(cut_res, config=None):
    tmp = {}
    for prev, nxt in pairwise(cut_res):
        if repl not in (prev, nxt):
            tmp.setdefault(prev, {})
            tmp[prev][nxt] = tmp[prev].get(nxt, 0) + 1
    for w, children in tmp.items():
        node = word_index_dic.setdefault(w, Wd_Node(w, config))
        for cw, cnt in children.items():
            cnode = word_index_dic.setdefault(cw, Wd_Node(cw, config))
            node.add_links(cnode, [cnt], direct='c')

